# MLflow performance export

Fetch completed Citi Bike training runs directly from MLflow, then export a monthly MAE chart for your deck. Run all cells in order.

The notebook reads MLflow without creating experiments or changing runs. On your laptop, use `http://localhost:5000`; in Docker Jupyter, the existing `MLFLOW_TRACKING_URI=http://mlflow:5000` environment variable is used. Requires `mlflow`, `pandas`, and `matplotlib`.

API reference: [MLflow client](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.client.html).

## Install notebook dependencies

Run this cell once in the selected notebook kernel. `%pip` installs into that kernel's environment. The MLflow version matches this project's tracking server. If imports still fail after installation, restart the kernel and run the remaining cells.

The import must be `from mlflow.tracking import MlflowClient` with no backslash before the dot.

In [ ]:
%pip install "mlflow==3.17.1" pandas matplotlib

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
from mlflow.tracking import MlflowClient

TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "citibike-demand-forecasting")
# Resolve exports consistently from either the repository root or notebooks directory.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yaml").exists()),
    Path.cwd(),
)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlflow_performance"
print(f"Tracking server: {TRACKING_URI}")
print(f"Exports: {OUTPUT_DIR}")

## Fetch runs

Keep all parameters, tags, and metrics in `runs_df`. Pagination ensures the export includes more than just the first page. If connection fails, start the MLflow service and check `TRACKING_URI` above.

In [ ]:
client = MlflowClient(tracking_uri=TRACKING_URI)
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f"Experiment {EXPERIMENT_NAME!r} not found at {TRACKING_URI}.")

records = []
page_token = None
while True:
    page = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="attributes.status = 'FINISHED'",
        max_results=1000,
        page_token=page_token,
    )
    for run in page:
        records.append({
            "run_id": run.info.run_id,
            "run_name": run.data.tags.get("mlflow.runName", run.info.run_id),
            "start_time": pd.to_datetime(run.info.start_time, unit="ms", utc=True),
            **{f"tags.{k}": v for k, v in run.data.tags.items()},
            **{f"params.{k}": v for k, v in run.data.params.items()},
            **{f"metrics.{k}": v for k, v in run.data.metrics.items()},
        })
    page_token = page.token
    if not page_token:
        break

runs_df = pd.DataFrame(records)
if runs_df.empty:
    raise ValueError("No completed runs found in this experiment.")
print(f"Fetched {len(runs_df)} completed runs.")
display(runs_df.head())

## Prepare monthly results

`trained_through` is both the final training cutoff and the end of the held-out test month in this project's training pipeline. Use the date tag, falling back to the run-name suffix for older runs. Execution time is not the historical test date.

Only runs with both test MAEs and a valid date are plotted. If a month has multiple runs, keep the most recently started run for that month; this is a reporting rule, not a claim that it was promoted or had the lowest error.

In [ ]:
def run_column(name):
    return runs_df.get(name, pd.Series(index=runs_df.index, dtype="object"))

cutoff = pd.to_datetime(run_column("tags.trained_through"), errors="coerce")
name_date = runs_df["run_name"].str.extract(r"(\d{4}-\d{2}-\d{2})$", expand=False)
cutoff = cutoff.fillna(pd.to_datetime(name_date, errors="coerce"))

performance_df = pd.DataFrame({
    "run_id": runs_df["run_id"],
    "run_name": runs_df["run_name"],
    "start_time": runs_df["start_time"],
    "test_month_end": cutoff,
    "model_type": run_column("tags.model_type"),
    "promoted": run_column("tags.promoted"),
    "model_mae": pd.to_numeric(run_column("metrics.test_model_mae"), errors="coerce"),
    "baseline_mae": pd.to_numeric(run_column("metrics.test_baseline_mae"), errors="coerce"),
})
usable = performance_df.dropna(subset=["test_month_end", "model_mae", "baseline_mae"]).copy()
usable["test_month"] = usable["test_month_end"].dt.to_period("M")
monthly_df = (
    usable.sort_values(["start_time", "run_id"])
    .drop_duplicates("test_month", keep="last")
    .sort_values("test_month_end")
    .reset_index(drop=True)
)
if monthly_df.empty:
    raise ValueError("No runs have dates and both test MAEs. Inspect runs_df.columns and your experiment name.")
monthly_df["improvement_pct"] = (
    100 * (monthly_df["baseline_mae"] - monthly_df["model_mae"])
    / monthly_df["baseline_mae"].where(monthly_df["baseline_mae"] != 0)
)
print(f"Skipped {len(performance_df) - len(usable)} runs missing dates or MAEs.")
print(f"Selected {len(monthly_df)} months from {len(usable)} eligible runs.")
display(monthly_df[["test_month_end", "model_type", "model_mae", "baseline_mae", "improvement_pct", "promoted"]])

## Chart and export

The selected model can change between months. This chart compares each month's candidate with its 7-day lag baseline on the same test data; it does not represent the deployed champion's historical performance. Lower MAE is better.

The PNG is saved at 300 dpi for slides, alongside an SVG and CSV exports. All exports go under the project's ignored `outputs/` directory.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
runs_df.to_csv(OUTPUT_DIR / "mlflow_runs.csv", index=False)
monthly_df.to_csv(OUTPUT_DIR / "monthly_performance.csv", index=False)

with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(12, 5.5), constrained_layout=True)
    ax.plot(monthly_df["test_month_end"], monthly_df["baseline_mae"],
            label="7-day lag baseline", color="#8a94a6", linestyle="--", marker="o", linewidth=2)
    ax.plot(monthly_df["test_month_end"], monthly_df["model_mae"],
            label="Monthly candidate model", color="#2563eb", marker="o", linewidth=2.5)
    ax.set(title="Monthly forecast error vs 7-day baseline", xlabel="Test month", ylabel="Test MAE (lower is better)")
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    for extension in ("png", "svg"):
        fig.savefig(OUTPUT_DIR / f"monthly_mae.{extension}", dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(path)